# 05 — Retrieval

Given a customer message, retrieve the most relevant past
`(customer_message, brand_response)` pairs so that `06_agent` can ground
reply generation in real precedents.

## Hard rules (read before running)

1. **Corpus is disjoint from dev and test — at root_id level AND at
   normalized-text level.** Enforced in Cell 4 as hard assertions.
   Non-negotiable.
2. **Test is evaluated once.** `TEST_LOCK.json` verifies hashes and refuses
   to re-run retrieval. Same discipline as `04`.
3. **`chosen_retriever.json` is immutable before test.** Written in Cell 15,
   read (never modified) in Cell 17.
4. **Test is not loaded until Cell 16.**
5. **Relevance = intent-match** (query's true intent == doc's intent).
   Imperfect proxy, documented. A 20-query human audit in Cell 14 provides
   a diagnostic sanity check — it does **not** change the winner.
6. **`response_text` = the first substantive brand response in the same
   `root_id`**, where "substantive" ∈ {resolution, informational,
   investigation}. Handoffs and empathy-only remain as metadata.
7. **Threads with no substantive response are excluded from the corpus**
   but logged to `excluded_no_substantive_response.csv`.
8. **Selection rule, declared before seeing dev:**
   - Primary metric: Recall@5
   - Tie-break (Δ ≤ 0.02): simpler / cheaper retriever
   - Diagnostic (does not affect choice): human audit, response-type mix

## What this notebook does not do

- Re-classify (that's `04`)
- Generate replies (that's `06`)
- Make escalation decisions (that's `06`)
- Use any test message as retrieval evidence

In [4]:
# ============ CELL 1b: Bootstrap (idempotent) ============
from pathlib import Path
import os, re, json, hashlib, sys, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def _find_root():
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = _find_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BRAND           = "GWRHelp"
CONFIG_DIR      = PROJECT_ROOT / "configs"
TAX_DIR         = PROJECT_ROOT / "runs" / "taxonomy"
GS_DIR          = PROJECT_ROOT / "runs" / "golden_set"
RUNS_CLASSIFIER = PROJECT_ROOT / "runs" / "classifier"
RUNS_RETRIEVAL  = PROJECT_ROOT / "runs" / "retrieval"
RUNS_RETRIEVAL.mkdir(parents=True, exist_ok=True)

from support_agent.retrieval.index import normalize_text, text_hash, Corpus

print("Bootstrap complete.")
print(f"  PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"  RUNS_RETRIEVAL: {RUNS_RETRIEVAL.relative_to(PROJECT_ROOT)}")

Bootstrap complete.
  PROJECT_ROOT:   D:\CODIN PLAYGROUND\ML-AI\ResolveIQ
  RUNS_RETRIEVAL: runs\retrieval


In [5]:
# ============ CELL 2: Verify frozen inputs ============
import yaml

INTENTS_YAML  = CONFIG_DIR / "intents.yaml"
INTENTS_SHA_F = TAX_DIR / "intents.yaml.sha256"
GOLDEN_JSONL  = GS_DIR / "golden_set.jsonl"
GOLDEN_SHA_F  = GS_DIR / "golden_set.sha256"
GOLDEN_META   = GS_DIR / "golden_set.meta.json"
CLASSIFIER_CHOSEN = RUNS_CLASSIFIER / "chosen.json"
CLASSIFIER_FINAL  = RUNS_CLASSIFIER / "final_metrics.json"
CLASSIFIER_PREDS  = RUNS_CLASSIFIER / "predictions_test.jsonl"
FIRST_INBOUND = PROJECT_ROOT / "runs" / "brand_selection" / "first_inbound_per_thread.pkl"
BRAND_RESPONSES = PROJECT_ROOT / "runs" / "brand_selection" / "brand_responses_classified.csv"

for p in (INTENTS_YAML, INTENTS_SHA_F, GOLDEN_JSONL, GOLDEN_SHA_F, GOLDEN_META,
          CLASSIFIER_CHOSEN, CLASSIFIER_FINAL, CLASSIFIER_PREDS,
          FIRST_INBOUND, BRAND_RESPONSES):
    assert p.exists(), f"Missing required input: {p}"

# Taxonomy
tax_sha = hashlib.sha256(INTENTS_YAML.read_bytes()).hexdigest()
assert tax_sha == INTENTS_SHA_F.read_text(encoding="utf-8").strip(), \
    "Taxonomy SHA mismatch"
tax = yaml.safe_load(INTENTS_YAML.read_bytes())
assert tax.get("draft") is False
INTENT_NAMES = [i["name"] for i in tax["intents"]]
OPERATIONAL_INTENTS = [n for n in INTENT_NAMES if n not in ("other", "ambiguous")]

# Golden
golden_sha = hashlib.sha256(GOLDEN_JSONL.read_bytes()).hexdigest()
golden_meta = json.loads(GOLDEN_META.read_text(encoding="utf-8"))
assert golden_sha == golden_meta["golden_sha256"]
assert golden_sha == GOLDEN_SHA_F.read_text(encoding="utf-8").strip()
assert golden_meta["taxonomy_sha256"] == tax_sha

# Classifier
chosen = json.loads(CLASSIFIER_CHOSEN.read_text(encoding="utf-8"))
chosen_sha = hashlib.sha256(CLASSIFIER_CHOSEN.read_bytes()).hexdigest()
assert chosen["taxonomy_sha256"] == tax_sha
assert chosen["golden_sha256"]   == golden_sha

# Classifier test predictions match golden test IDs (set equality)
golden_test_ids = set(int(x) for x in golden_meta["test_root_ids"])
classifier_test_ids = set()
with open(CLASSIFIER_PREDS, "r", encoding="utf-8") as f:
    for line in f:
        classifier_test_ids.add(int(json.loads(line)["root_id"]))
assert classifier_test_ids == golden_test_ids, (
    f"Classifier test IDs do not match golden test IDs. "
    f"Only in classifier: {len(classifier_test_ids - golden_test_ids)}, "
    f"only in golden: {len(golden_test_ids - classifier_test_ids)}"
)

# Brand responses need expected columns
resp_cols = pd.read_csv(BRAND_RESPONSES, nrows=0).columns.tolist()
print(f"brand_responses_classified.csv columns: {resp_cols}")

print(f"Taxonomy SHA:     {tax_sha[:16]}...")
print(f"Golden SHA:       {golden_sha[:16]}...")
print(f"Chosen cls:       {chosen['approach']}  (SHA {chosen_sha[:12]}...)")
print(f"Golden test IDs:  {len(golden_test_ids)}")
print(f"Classifier test IDs match: ✓")

brand_responses_classified.csv columns: ['tweet_id', 'author_id', 'text', 'case_type']
Taxonomy SHA:     7a05e4af68a50981...
Golden SHA:       acde341dc09e85ae...
Chosen cls:       llm_zeroshot  (SHA 7d51251a172e...)
Golden test IDs:  139
Classifier test IDs match: ✓


In [6]:
# ============ CELL 3: Load DEV only ============
TEST_ROOT_IDS = set(int(x) for x in golden_meta["test_root_ids"])

dev_rows = []
with open(GOLDEN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["split"] == "dev":
            dev_rows.append(r)
dev_df = pd.DataFrame(dev_rows)
DEV_ROOT_IDS = set(dev_df["root_id"].astype(int))

assert len(dev_df) == golden_meta["dev_size"]
assert set(dev_df["true_intent"].unique()) <= set(INTENT_NAMES)

# Hash test texts WITHOUT loading them for inspection
# We need only the hashes for the corpus exclusion check.
with open(GOLDEN_JSONL, "r", encoding="utf-8") as f:
    test_text_hashes = set()
    for line in f:
        r = json.loads(line)
        if r["split"] == "test":
            test_text_hashes.add(text_hash(r["customer_text"]))

dev_text_hashes = set(dev_df["customer_text"].map(text_hash))

print(f"Dev rows:             {len(dev_df)}")
print(f"Test root_ids:        {len(TEST_ROOT_IDS)} (labels not read)")
print(f"Test text hashes:     {len(test_text_hashes)}")
print(f"Dev text hashes:      {len(dev_text_hashes)}")
print(dev_df["true_intent"].value_counts().to_string())

Dev rows:             59
Test root_ids:        139 (labels not read)
Test text hashes:     139
Dev text hashes:      59
true_intent
delay_compensation    15
service_disruption     8
on_board_issue         7
refund_request         6
seat_reservation       5
praise_or_chatter      5
other                  5
timetable_info         4
booking_issue          3
lost_property          1


In [7]:
# ============ CELL 4: Build primary retrieval benchmark corpus ============
# Source: training pool from 04 (coverage + human-verified + examples)
# Response extraction: first SUBSTANTIVE brand response per root_id
# Exclusions: all golden root_ids and normalized-text hashes

COVERAGE_CSV   = TAX_DIR / "coverage_labeling_sheet.csv"
VERIFICATION   = GS_DIR / "verification_sheet.csv"
EXAMPLES_CSV   = TAX_DIR / "intent_examples.csv"

# --- Build label pool (same three sources as 04) ---
coverage = pd.read_csv(COVERAGE_CSV, encoding="utf-8").rename(
    columns={"intent": "true_intent"}
)[["root_id", "customer_text", "true_intent"]]
coverage["_source"] = "coverage"

verif = pd.read_csv(VERIFICATION, encoding="utf-8")
verif = verif[verif["human_confirmed"].astype(str).str.lower() == "y"]
human = verif[["root_id", "customer_text", "true_intent"]].copy()
human["_source"] = "human_pool"

examples = pd.read_csv(EXAMPLES_CSV, encoding="utf-8")
examples = examples[examples["_keep"].astype(str).str.lower() == "y"]
ex = examples[["root_id", "customer_text", "intent"]].rename(
    columns={"intent": "true_intent"}
)
ex["_source"] = "human_examples"

labels = pd.concat([coverage, human, ex], ignore_index=True)
labels = labels.drop_duplicates(subset=["root_id"])
labels["true_intent"] = labels["true_intent"].astype(str)
labels = labels[labels["true_intent"].isin(INTENT_NAMES)]

# --- Load responses ---
responses = pd.read_csv(BRAND_RESPONSES, encoding="utf-8")
print(f"Response file columns: {responses.columns.tolist()}")
print(f"Response file rows:    {len(responses)}")
display(responses.head(3))

Response file columns: ['tweet_id', 'author_id', 'text', 'case_type']
Response file rows:    183976


,tweet_id,author_id,text,case_type
0,10,sprintcare,@115712 Hello! We never like our customers to ...,other
1,9,sprintcare,@115712 I would love the chance to review the ...,other
2,6,sprintcare,@115712 Can you please send us a private messa...,handoff_dm


In [8]:
# ============ DIAGNOSTIC ============
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
resp = pd.read_csv(PROJECT_ROOT / "runs/brand_selection/brand_responses_classified.csv")
print("Response file:")
print(f"  Rows: {len(resp)}")
print(f"  Columns: {resp.columns.tolist()}")
print(f"\ncase_type values:")
print(resp["case_type"].value_counts().to_string())
print(f"\nSample:")
display(resp.head(3))

# Now look for thread linkage files
for candidate in [
    PROJECT_ROOT / "data/interim/threads_sample.pkl",
    PROJECT_ROOT / "runs/brand_selection/first_inbound_per_thread.pkl",
]:
    if candidate.exists():
        df = pd.read_pickle(candidate)
        print(f"\n{candidate.name}:")
        print(f"  Rows: {len(df)}")
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head(2))

Response file:
  Rows: 183976
  Columns: ['tweet_id', 'author_id', 'text', 'case_type']

case_type values:
case_type
other            108097
link_handoff      30758
handoff_dm        26068
informational      6085
info_request       5880
empathy_only       5585
investigation      1181
resolution          322

Sample:


,tweet_id,author_id,text,case_type
0,10,sprintcare,@115712 Hello! We never like our customers to ...,other
1,9,sprintcare,@115712 I would love the chance to review the ...,other
2,6,sprintcare,@115712 Can you please send us a private messa...,handoff_dm



threads_sample.pkl:
  Rows: 405766
  Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'root_id', 'valid_thread']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,root_id,valid_thread
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0,8,True
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0,8,True



first_inbound_per_thread.pkl:
  Rows: 118101
  Columns: ['root_id', 'first_inbound_tweet_id', 'customer_author_id', 'customer_text', 'created_at', 'n_tweets', 'n_inbound', 'n_outbound', 'brand_author_id']


,root_id,first_inbound_tweet_id,customer_author_id,customer_text,created_at,n_tweets,n_inbound,n_outbound,brand_author_id
0,8,8,115712,@sprintcare is the worst customer service,2017-10-31 21:45:10+00:00,10,5,5,sprintcare
1,20,20,115715,"@115714 whenever I contact customer support, t...",2017-10-31 22:03:34+00:00,2,1,1,sprintcare


In [9]:
# ============ CELL 4b: Attach responses + filter + hash ============
# Response file schema: tweet_id, author_id, text, case_type
# Thread index:         tweet_id, root_id, created_at (in threads_sample.pkl)

THREADS_PATH = PROJECT_ROOT / "data" / "interim" / "threads_sample.pkl"
assert THREADS_PATH.exists(), f"Missing thread index: {THREADS_PATH}"

# --- Load and inspect responses ---
responses = pd.read_csv(BRAND_RESPONSES, encoding="utf-8")
print(f"Loaded {len(responses):,} classified responses")
print(f"Top author_id values (first 10):")
print(responses["author_id"].astype(str).value_counts().head(10).to_string())

# --- Filter to GWRHelp only (case-insensitive) ---
responses["_author_str"] = responses["author_id"].astype(str)
responses = responses[responses["_author_str"].str.lower() == BRAND.lower()].copy()
print(f"\n{BRAND} responses: {len(responses):,}")
assert len(responses) > 0, \
    f"No responses for {BRAND}. Check author_id values above."

# --- Load thread index (tweet_id → root_id + created_at) ---
threads = pd.read_pickle(THREADS_PATH)
print(f"\nThread index rows: {len(threads):,}")
print(f"Thread index columns: {threads.columns.tolist()}")

threads_index = (
    threads[["tweet_id", "root_id", "created_at"]]
    .drop_duplicates("tweet_id")
    .copy()
)
# Normalize types for join
threads_index["tweet_id"] = threads_index["tweet_id"].astype("int64")
responses["tweet_id"]      = responses["tweet_id"].astype("int64")

# --- Join responses → root_id ---
responses = responses.merge(threads_index, on="tweet_id", how="left")
missing_thread = int(responses["root_id"].isna().sum())
join_rate = 1 - missing_thread / len(responses) if len(responses) else 0
print(f"\nResponses joined to a thread: "
      f"{len(responses) - missing_thread:,} / {len(responses):,} "
      f"({join_rate:.1%})")

if join_rate < 0.80:
    print("WARNING: Low join rate. The response file may cover a different "
          "sample than threads_sample.pkl. Investigate before proceeding.")

responses = responses.dropna(subset=["root_id"]).copy()
responses["root_id"] = responses["root_id"].astype(int)

# --- Parse timestamps for chronological ordering ---
responses["created_at"] = pd.to_datetime(
    responses["created_at"], errors="coerce", utc=True
)
responses = responses.dropna(subset=["created_at"])

# --- Canonical column names ---
responses = responses.rename(columns={
    "text":      "response_text",
    "case_type": "response_type",
})

print(f"\nResponse-type distribution ({BRAND} only):")
print(responses["response_type"].value_counts().to_string())

# --- Pick the first SUBSTANTIVE response per root_id ---
SUBSTANTIVE = {"resolution", "informational", "investigation"}
substantive = responses[responses["response_type"].isin(SUBSTANTIVE)].copy()
print(f"\nSubstantive responses: {len(substantive):,} "
      f"({len(substantive) / len(responses):.1%} of all {BRAND} responses)")

substantive = substantive.sort_values("created_at")
picked = (substantive
          .drop_duplicates(subset=["root_id"], keep="first")
          [["root_id", "response_text", "response_type", "created_at"]])

print(f"Threads with a substantive response: {len(picked):,}")

# --- Merge with labels ---
merged = labels.merge(picked, on="root_id", how="left")
print(f"\nLabels:                          {len(labels):,}")
print(f"After merging substantive resps: {merged['response_text'].notna().sum():,}")

# --- Exclusions: golden (root + text) ---
golden_root_ids = DEV_ROOT_IDS | TEST_ROOT_IDS
merged = merged[~merged["root_id"].astype(int).isin(golden_root_ids)]
print(f"After excluding golden root_ids: {len(merged):,}")

merged["_text_hash"] = merged["customer_text"].map(text_hash)
merged = merged[~merged["_text_hash"].isin(dev_text_hashes)]
merged = merged[~merged["_text_hash"].isin(test_text_hashes)]
print(f"After excluding golden texts:    {len(merged):,}")

# --- Log no-substantive-response threads ---
no_response = merged[merged["response_text"].isna()].copy()
no_response["exclusion_reason"] = "no_substantive_response"
no_response[["root_id", "customer_text"]].to_csv(
    RUNS_RETRIEVAL / "excluded_no_substantive_response.csv",
    index=False, quoting=1, encoding="utf-8",
)
print(f"\nExcluded (no substantive response): {len(no_response):,}")

corpus_df = merged.dropna(subset=["response_text"]).reset_index(drop=True)
corpus_df["normalized"] = corpus_df["customer_text"].map(normalize_text)

# --- Disjointness asserts (belt and braces) ---
corpus_root_ids = set(corpus_df["root_id"].astype(int))
assert not (corpus_root_ids & DEV_ROOT_IDS),  "Corpus leaks dev root_ids"
assert not (corpus_root_ids & TEST_ROOT_IDS), "Corpus leaks test root_ids"
assert not (set(corpus_df["_text_hash"]) & dev_text_hashes),  "Corpus leaks dev texts"
assert not (set(corpus_df["_text_hash"]) & test_text_hashes), "Corpus leaks test texts"

# --- Build Corpus ---
corpus = Corpus(
    doc_ids         = corpus_df["root_id"].to_numpy(dtype=int),
    customer_texts  = corpus_df["customer_text"].astype(str).tolist(),
    normalized      = corpus_df["normalized"].tolist(),
    intents         = corpus_df["true_intent"].astype(str).to_numpy(),
    response_texts  = corpus_df["response_text"].astype(str).tolist(),
    response_types  = corpus_df["response_type"].astype(str).to_numpy(),
    source          = corpus_df["_source"].astype(str).to_numpy(),
)

CORPUS_JSONL = RUNS_RETRIEVAL / "corpus.jsonl"
corpus_sha = corpus.save_jsonl(CORPUS_JSONL)
(RUNS_RETRIEVAL / "corpus.sha256").write_text(corpus_sha + "\n", encoding="utf-8")

print(f"\n{'='*50}")
print(f"Corpus size:      {len(corpus):,}")
print(f"Corpus SHA:       {corpus_sha[:16]}...")
print(f"{'='*50}")
print(f"\nIntent distribution:")
print(pd.Series(corpus.intents).value_counts().to_string())
print(f"\nResponse-type distribution:")
print(pd.Series(corpus.response_types).value_counts().to_string())
print(f"\nSource distribution:")
print(pd.Series(corpus.source).value_counts().to_string())

Loaded 183,976 classified responses
Top author_id values (first 10):
author_id
AmazonHelp         24775
AppleSupport       16065
Uber_Support        8293
SpotifyCares        6505
Delta               6458
Tesco               5638
AmericanAir         5289
comcastcares        4924
TMobileHelp         4911
British_Airways     4304

GWRHelp responses: 2,794

Thread index rows: 405,766
Thread index columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'root_id', 'valid_thread']

Responses joined to a thread: 2,794 / 2,794 (100.0%)


C:\Users\adity\AppData\Local\Temp\ipykernel_29616\2326813319.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  responses["created_at"] = pd.to_datetime(



Response-type distribution (GWRHelp only):
response_type
other            2413
empathy_only      106
informational     101
link_handoff       92
resolution         47
info_request       24
handoff_dm          9
investigation       2

Substantive responses: 150 (5.4% of all GWRHelp responses)
Threads with a substantive response: 140

Labels:                          685
After merging substantive resps: 69
After excluding golden root_ids: 487
After excluding golden texts:    487

Excluded (no substantive response): 437

Corpus size:      50
Corpus SHA:       69327018136200df...

Intent distribution:
service_disruption    15
praise_or_chatter      6
on_board_issue         6
booking_issue          6
delay_compensation     5
seat_reservation       4
timetable_info         3
other                  3
refund_request         2

Response-type distribution:
informational    31
resolution       18
investigation     1

Source distribution:
coverage          29
human_pool        11
human_examples  

In [10]:
# ============ CELL 5: Metric utilities ============
from sklearn.metrics import f1_score

def recall_at_k(retrieved_ids: list[int],
                relevant_ids: set[int],
                k: int) -> float:
    topk = retrieved_ids[:k]
    return 1.0 if any(d in relevant_ids for d in topk) else 0.0


def reciprocal_rank(retrieved_ids: list[int],
                    relevant_ids: set[int]) -> float:
    for rank, d in enumerate(retrieved_ids, start=1):
        if d in relevant_ids:
            return 1.0 / rank
    return 0.0


def evaluate_retriever_on_queries(retriever,
                                  queries_df: pd.DataFrame,
                                  corpus: Corpus,
                                  k_list=(1, 3, 5, 10),
                                  query_filter=None):
    """Run a retriever on every query. Returns per-query records."""
    doc_id_to_idx = corpus.doc_id_to_index()
    # Precompute relevant doc_ids per intent
    intent_docs: dict[str, set[int]] = {}
    for intent in set(corpus.intents):
        intent_docs[intent] = {int(corpus.doc_ids[i])
                               for i in range(len(corpus))
                               if corpus.intents[i] == intent}

    records = []
    for _, q in queries_df.iterrows():
        pred = retriever.retrieve(q["customer_text"], k=max(k_list))
        retrieved_ids = [d for d, _ in pred]
        scores = [s for _, s in pred]
        true_intent = q["true_intent"]
        relevant = intent_docs.get(true_intent, set())
        coverage_flag = 1 if relevant else 0

        rec = {
            "root_id": int(q["root_id"]),
            "true_intent": true_intent,
            "_source": q.get("_source", "unknown"),
            "corpus_coverage": coverage_flag,
            "top_ids": retrieved_ids,
            "top_scores": scores,
            "mrr": reciprocal_rank(retrieved_ids, relevant),
        }
        for k in k_list:
            rec[f"recall@{k}"] = recall_at_k(retrieved_ids, relevant, k)
        records.append(rec)
    return pd.DataFrame(records)


def aggregate_metrics(per_query: pd.DataFrame, k_list=(1, 3, 5, 10)):
    """Roll up per-query records into headline metrics."""
    out = {}
    for k in k_list:
        col = f"recall@{k}"
        out[f"recall@{k}"] = float(per_query[col].mean())
        if per_query["corpus_coverage"].sum() > 0:
            covered = per_query[per_query["corpus_coverage"] == 1]
            out[f"recall@{k}_cov"] = float(covered[col].mean())
        else:
            out[f"recall@{k}_cov"] = None
    out["mrr"] = float(per_query["mrr"].mean())
    out["coverage"] = float(per_query["corpus_coverage"].mean())
    return out


def bootstrap_ci(values: np.ndarray, n_boot: int = 1000, seed: int = 42):
    rng = np.random.RandomState(seed)
    n = len(values)
    if n == 0:
        return (0.0, 0.0)
    scores = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.randint(0, n, n)
        scores[b] = values[idx].mean()
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))


def response_type_mix(per_query: pd.DataFrame, corpus: Corpus, k: int = 5):
    """Fraction of top-k docs per response_type. Grounding diagnostic."""
    doc_id_to_idx = corpus.doc_id_to_index()
    counts = {}
    total = 0
    for _, r in per_query.iterrows():
        for doc_id in r["top_ids"][:k]:
            idx = doc_id_to_idx.get(int(doc_id))
            if idx is None:
                continue
            rt = str(corpus.response_types[idx])
            counts[rt] = counts.get(rt, 0) + 1
            total += 1
    if total == 0:
        return {}
    return {rt: c / total for rt, c in counts.items()}


print("Metric utilities loaded.")

Metric utilities loaded.


In [11]:
# ============ CELL 6: Random baseline (20 seeds) ============
from support_agent.retrieval.retriever import RandomRetriever

seeds = [RANDOM_STATE + i for i in range(20)]
random_results = []
for s in seeds:
    r = RandomRetriever(corpus, seed=s)
    pq = evaluate_retriever_on_queries(r, dev_df, corpus)
    agg = aggregate_metrics(pq)
    agg["seed"] = s
    random_results.append(agg)

random_df = pd.DataFrame(random_results)

# Aggregate across seeds
random_summary = {
    "approach": "random",
    "recall@1": float(random_df["recall@1"].mean()),
    "recall@3": float(random_df["recall@3"].mean()),
    "recall@5": float(random_df["recall@5"].mean()),
    "recall@10": float(random_df["recall@10"].mean()),
    "mrr": float(random_df["mrr"].mean()),
    "coverage": float(random_df["coverage"].mean()),
    "recall@5_95ci_lo": float(np.percentile(random_df["recall@5"], 2.5)),
    "recall@5_95ci_hi": float(np.percentile(random_df["recall@5"], 97.5)),
    "n_seeds": len(seeds),
}
print(f"Random baseline (mean across {len(seeds)} seeds):")
for k, v in random_summary.items():
    if isinstance(v, float):
        print(f"  {k:20s} {v:.4f}")
    else:
        print(f"  {k:20s} {v}")

Random baseline (mean across 20 seeds):
  approach             random
  recall@1             0.1347
  recall@3             0.3102
  recall@5             0.4415
  recall@10            0.6746
  mrr                  0.2682
  coverage             0.9831
  recall@5_95ci_lo     0.3462
  recall@5_95ci_hi     0.5610
  n_seeds              20


In [12]:
# ============ CELL 7: TF-IDF ============
from support_agent.retrieval.retriever import TFIDFRetriever

def run_and_save(name, retriever, queries_df, out_dir):
    out_dir = RUNS_RETRIEVAL / name
    out_dir.mkdir(parents=True, exist_ok=True)

    pq = evaluate_retriever_on_queries(retriever, queries_df, corpus)
    agg = aggregate_metrics(pq)

    # Record response-type mix as grounding diagnostic
    rt_mix = response_type_mix(pq, corpus, k=5)

    # Persist
    pq.to_json(out_dir / "predictions_dev.jsonl",
               orient="records", lines=True, force_ascii=False)
    (out_dir / "metrics_dev.json").write_text(
        json.dumps({**agg, "response_type_mix_top5": rt_mix}, indent=2),
        encoding="utf-8"
    )
    (out_dir / "meta.json").write_text(json.dumps({
        "approach": name,
        "params": getattr(retriever, "params", {}),
        "taxonomy_sha256": tax_sha,
        "golden_sha256": golden_sha,
        "corpus_sha256": corpus_sha,
        "normalization_version": corpus.normalization_version,
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }, indent=2), encoding="utf-8")

    print(f"[{name}] recall@5={agg['recall@5']:.3f} "
          f"mrr={agg['mrr']:.3f} coverage={agg['coverage']:.3f}")
    return agg, rt_mix

tfidf = TFIDFRetriever(corpus, ngram_range=(1, 2), min_df=1)
tfidf_agg, tfidf_rt = run_and_save("tfidf", tfidf, dev_df, RUNS_RETRIEVAL)

[tfidf] recall@5=0.746 mrr=0.493 coverage=0.983


In [13]:
# ============ CELL 8: BM25 ============
from support_agent.retrieval.retriever import BM25Retriever

bm25 = BM25Retriever(corpus, k1=1.5, b=0.75)
bm25_agg, bm25_rt = run_and_save("bm25", bm25, dev_df, RUNS_RETRIEVAL)

[bm25] recall@5=0.661 mrr=0.455 coverage=0.983


In [14]:
# ============ CELL 9: Embedding ============
from support_agent.retrieval.retriever import EmbeddingRetriever

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
emb = EmbeddingRetriever(corpus, model_name=EMBEDDING_MODEL)
emb_agg, emb_rt = run_and_save("embedding", emb, dev_df, RUNS_RETRIEVAL)

d:\CODIN PLAYGROUND\ML-AI\ResolveIQ\resolveIQ\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4167.80it/s]


[embedding] recall@5=0.576 mrr=0.429 coverage=0.983


In [15]:
# ============ CELL 10: Hybrid RRF (BM25 + Embedding) ============
from support_agent.retrieval.retriever import RRFRetriever

hybrid = RRFRetriever(corpus, [bm25, emb], k_rrf=60)
hybrid_agg, hybrid_rt = run_and_save("hybrid_rrf", hybrid, dev_df, RUNS_RETRIEVAL)

[hybrid_rrf] recall@5=0.661 mrr=0.498 coverage=0.983


In [16]:
# ============ CELL 11: Oracle-intent BM25 (diagnostic) ============
from support_agent.retrieval.retriever import IntentConditionedRetriever
# Uses the TRUE intent (from dev labels) to filter the corpus.
# This is diagnostic only — never part of the production pipeline.

def oracle_intent_resolver(query_text, *, root_id=None, true_intent=None):
    return true_intent

oracle = IntentConditionedRetriever(
    corpus, bm25, oracle_intent_resolver,
    name="oracle_intent_bm25",
)

# IntentConditionedRetriever takes an extra kwarg on retrieve(). We wrap
# the evaluation to pass true_intent per query.
def evaluate_intent_conditioned(retriever, queries_df, corpus, k_list=(1,3,5,10)):
    records = []
    intent_docs = {}
    for intent in set(corpus.intents):
        intent_docs[intent] = {int(corpus.doc_ids[i])
                               for i in range(len(corpus))
                               if corpus.intents[i] == intent}
    for _, q in queries_df.iterrows():
        hits = retriever.retrieve(
            q["customer_text"], k=max(k_list),
            root_id=int(q["root_id"]),
            true_intent=q["true_intent"],
        )
        retrieved_ids = [d for d, _ in hits]
        relevant = intent_docs.get(q["true_intent"], set())
        rec = {
            "root_id": int(q["root_id"]),
            "true_intent": q["true_intent"],
            "_source": q.get("_source", "unknown"),
            "corpus_coverage": 1 if relevant else 0,
            "top_ids": retrieved_ids,
            "top_scores": [s for _, s in hits],
            "mrr": reciprocal_rank(retrieved_ids, relevant),
        }
        for k in k_list:
            rec[f"recall@{k}"] = recall_at_k(retrieved_ids, relevant, k)
        records.append(rec)
    return pd.DataFrame(records)

def run_intent_conditioned(name, retriever, queries_df):
    out_dir = RUNS_RETRIEVAL / name
    out_dir.mkdir(parents=True, exist_ok=True)
    pq = evaluate_intent_conditioned(retriever, queries_df, corpus)
    agg = aggregate_metrics(pq)
    rt_mix = response_type_mix(pq, corpus, k=5)
    pq.to_json(out_dir / "predictions_dev.jsonl",
               orient="records", lines=True, force_ascii=False)
    (out_dir / "metrics_dev.json").write_text(
        json.dumps({**agg, "response_type_mix_top5": rt_mix}, indent=2),
        encoding="utf-8"
    )
    (out_dir / "meta.json").write_text(json.dumps({
        "approach": name,
        "diagnostic_only": True,
        "taxonomy_sha256": tax_sha,
        "golden_sha256": golden_sha,
        "corpus_sha256": corpus_sha,
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }, indent=2), encoding="utf-8")
    print(f"[{name}] recall@5={agg['recall@5']:.3f} mrr={agg['mrr']:.3f}")
    return agg, rt_mix

oracle_agg, oracle_rt = run_intent_conditioned("oracle_intent_bm25", oracle, dev_df)

[oracle_intent_bm25] recall@5=0.966 mrr=0.966


In [17]:
# ============ CELL 12: Predicted-intent BM25 ============
# Uses the chosen classifier's dev predictions to filter the corpus.

from support_agent.retrieval.retriever import IntentConditionedRetriever

# Resolve the chosen approach from 04's lock
_chosen_path = RUNS_CLASSIFIER / "chosen.json"
assert _chosen_path.exists(), f"Run 04 first. Missing: {_chosen_path}"
_chosen = json.loads(_chosen_path.read_text(encoding="utf-8"))
_chosen_approach = _chosen["approach"]

# Load that approach's dev predictions
CLASSIFIER_DEV = RUNS_CLASSIFIER / _chosen_approach / "predictions_dev.jsonl"
assert CLASSIFIER_DEV.exists(), \
    f"Missing dev predictions for chosen approach '{_chosen_approach}': {CLASSIFIER_DEV}"

pred_dev = pd.read_json(CLASSIFIER_DEV, orient="records", lines=True)
print(f"Chosen classifier: {_chosen_approach}")
print(f"Dev predictions:   {len(pred_dev)} rows")
print(f"Columns:           {pred_dev.columns.tolist()}")

# Verify the rows cover exactly the dev queries
assert set(pred_dev["root_id"].astype(int)) == DEV_ROOT_IDS, (
    "Classifier dev predictions do not cover the same root_ids as dev_df"
)
assert len(pred_dev) == len(dev_df), \
    f"Row count mismatch: predictions={len(pred_dev)}, dev={len(dev_df)}"

pred_map = dict(zip(pred_dev["root_id"].astype(int), pred_dev["pred_intent"]))

def classifier_intent_resolver(query_text, *, root_id=None, true_intent=None):
    if root_id is None:
        return "other"
    return pred_map.get(int(root_id), "other")

pred_intent = IntentConditionedRetriever(
    corpus, bm25, classifier_intent_resolver,
    name="predicted_intent_bm25",
)
pred_intent_agg, pred_intent_rt = run_intent_conditioned(
    "predicted_intent_bm25", pred_intent, dev_df
)

Chosen classifier: llm_zeroshot
Dev predictions:   59 rows
Columns:           ['root_id', 'true_intent', 'pred_intent', 'confidence', 'alternative_intent', 'alternative_confidence', 'split', '_source']
[predicted_intent_bm25] recall@5=0.610 mrr=0.610


In [18]:
# ============ CELL 13: Dev comparison ============
approaches = [
    ("random", random_summary),
    ("tfidf", tfidf_agg),
    ("bm25", bm25_agg),
    ("embedding", emb_agg),
    ("hybrid_rrf", hybrid_agg),
    ("oracle_intent_bm25", oracle_agg),
    ("predicted_intent_bm25", pred_intent_agg),
]

rows = []
for name, agg in approaches:
    rows.append({
        "approach":     name,
        "recall@1":     agg["recall@1"],
        "recall@3":     agg["recall@3"],
        "recall@5":     agg["recall@5"],
        "recall@10":    agg["recall@10"],
        "recall@5_cov": agg.get("recall@5_cov"),
        "mrr":          agg["mrr"],
        "coverage":     agg["coverage"],
    })

comparison = pd.DataFrame(rows).sort_values("recall@5", ascending=False).reset_index(drop=True)
display(comparison.round(4))

# Save
comparison.to_csv(RUNS_RETRIEVAL / "comparison_dev.csv", index=False, quoting=1)

# --- Winner selection (predeclared rule) ---
# Primary: recall@5. Tie-break within Δ 0.02: simpler/cheaper.
candidates = [c for c in comparison.to_dict("records")
              if c["approach"] not in ("oracle_intent_bm25",)]  # diagnostic excluded from winner
winner_row = candidates[0]  # highest recall@5 (sorted)
top_score = winner_row["recall@5"]
within_delta = [c for c in candidates if top_score - c["recall@5"] <= 0.02]

# Prefer simpler/cheaper if multiple are within delta
# Precedence: bm25 < tfidf < embedding < hybrid_rrf < predicted_intent_bm25
precedence = ["bm25", "tfidf", "embedding", "hybrid_rrf", "predicted_intent_bm25", "random"]
chosen_name = None
for p in precedence:
    for c in within_delta:
        if c["approach"] == p:
            chosen_name = p
            break
    if chosen_name:
        break

print(f"\nTop recall@5: {winner_row['approach']} = {top_score:.4f}")
print(f"Within Δ 0.02: {[c['approach'] for c in within_delta]}")
print(f"Chosen by predeclared rule (primary + simplicity tie-break): {chosen_name}")

WINNER = chosen_name
CHOSEN_METRICS = next(c for c in candidates if c["approach"] == WINNER)
print(f"\nWINNER metrics: {CHOSEN_METRICS}")

,approach,recall@1,recall@3,recall@5,recall@10,recall@5_cov,mrr,coverage
0,oracle_intent_bm25,0.9661,0.9661,0.9661,0.9661,0.9828,0.9661,0.9831
1,tfidf,0.3390,0.5763,0.7458,0.8475,0.7586,0.4930,0.9831
2,bm25,0.3051,0.5424,0.6610,0.8136,0.6724,0.4551,0.9831
3,hybrid_rrf,0.3559,0.5593,0.6610,0.8644,0.6724,0.4982,0.9831
4,predicted_intent_bm25,0.6102,0.6102,0.6102,0.6102,0.6207,0.6102,0.9831
5,embedding,0.2881,0.5254,0.5763,0.7966,0.5862,0.4287,0.9831
6,random,0.1347,0.3102,0.4415,0.6746,NaN,0.2682,0.9831



Top recall@5: tfidf = 0.7458
Within Δ 0.02: ['tfidf']
Chosen by predeclared rule (primary + simplicity tie-break): tfidf

WINNER metrics: {'approach': 'tfidf', 'recall@1': 0.3389830508474576, 'recall@3': 0.576271186440678, 'recall@5': 0.7457627118644068, 'recall@10': 0.847457627118644, 'recall@5_cov': 0.7586206896551724, 'mrr': 0.4929580306698951, 'coverage': 0.9830508474576272}


In [19]:
print(f"Corpus size: {len(corpus)}")
print(f"Intent distribution:")
print(pd.Series(corpus.intents).value_counts().to_string())

Corpus size: 50
Intent distribution:
service_disruption    15
praise_or_chatter      6
on_board_issue         6
booking_issue          6
delay_compensation     5
seat_reservation       4
timetable_info         3
other                  3
refund_request         2


In [20]:
# ============ CELL 14: Human relevance audit (DIAGNOSTIC ONLY) ============
# Stratified 20 dev queries (10 natural + 10 targeted).
# Top-5 of the chosen retriever are exported for manual labeling.
# This does NOT change the winner. It provides a sanity check that the
# automated recall@5 metric correlates directionally with usefulness.

import random

rng = random.Random(RANDOM_STATE)

natural = dev_df[dev_df["_source"] == "natural"].sample(
    n=min(10, len(dev_df[dev_df["_source"] == "natural"])),
    random_state=RANDOM_STATE,
)
targeted = dev_df[dev_df["_source"].astype(str).str.startswith("targeted:")].sample(
    n=min(10, len(dev_df[dev_df["_source"].astype(str).str.startswith("targeted:")])),
    random_state=RANDOM_STATE,
)
audit_queries = pd.concat([natural, targeted]).drop_duplicates("root_id")

# Pull top-5 from the chosen retriever
winner_retriever = {
    "tfidf": tfidf, "bm25": bm25, "embedding": emb,
    "hybrid_rrf": hybrid, "random": RandomRetriever(corpus, seed=RANDOM_STATE),
    "predicted_intent_bm25": pred_intent,
}[WINNER]

doc_id_to_idx = corpus.doc_id_to_index()
rows = []
for _, q in audit_queries.iterrows():
    hits = winner_retriever.retrieve(q["customer_text"], k=5)
    for rank, (doc_id, score) in enumerate(hits, start=1):
        idx = doc_id_to_idx.get(int(doc_id))
        if idx is None:
            continue
        rows.append({
            "query_root_id":    int(q["root_id"]),
            "query_text":       q["customer_text"],
            "query_intent":     q["true_intent"],
            "rank":             rank,
            "doc_root_id":      int(doc_id),
            "doc_text":         corpus.customer_texts[idx],
            "doc_response":     corpus.response_texts[idx],
            "doc_response_type": str(corpus.response_types[idx]),
            "doc_intent":       str(corpus.intents[idx]),
            "score":            float(score),
            "grade":            "",   # 0 / 1 / 2 — to fill in manually
            "notes":            "",
        })

AUDIT_CSV = RUNS_RETRIEVAL / "human_relevance_audit.csv"
pd.DataFrame(rows).to_csv(AUDIT_CSV, index=False, quoting=1, encoding="utf-8")
print(f"Wrote {AUDIT_CSV}  ({len(rows)} judgments)")
print("\nLabeling guide:")
print("  2 = directly useful precedent")
print("  1 = related but incomplete")
print("  0 = not useful")
print("\nThis audit is diagnostic only. It does NOT change the chosen retriever.")

Wrote D:\CODIN PLAYGROUND\ML-AI\ResolveIQ\runs\retrieval\human_relevance_audit.csv  (100 judgments)

Labeling guide:
  2 = directly useful precedent
  1 = related but incomplete
  0 = not useful

This audit is diagnostic only. It does NOT change the chosen retriever.


In [22]:
# ============ CELL 14b: Aggregate human audit ============
import math

audit = pd.read_csv(RUNS_RETRIEVAL / "human_relevance_audit.csv", encoding="utf-8")
graded = audit[audit["grade"].notna() & (audit["grade"].astype(str) != "")].copy()

if len(graded) == 0:
    print("No grades filled yet. Fill 'grade' column in human_relevance_audit.csv, then rerun.")
else:
    graded["grade"] = graded["grade"].astype(int)

    def dcg(gains):
        return sum(g / math.log2(i + 2) for i, g in enumerate(gains))

    ndcgs, precisions = [], []
    for qid, group in graded.groupby("query_root_id"):
        gains = group.sort_values("rank")["grade"].tolist()
        ideal = sorted(gains, reverse=True)
        d = dcg(gains)
        i = dcg(ideal)
        ndcgs.append(d / i if i > 0 else 0.0)
        precisions.append(sum(1 for g in gains if g >= 1) / len(gains))

    print(f"Human relevance audit — {len(graded['query_root_id'].unique())} queries, "
          f"{len(graded)} judgments")
    print(f"  nDCG@5:           {np.mean(ndcgs):.3f}")
    print(f"  Graded P@5 (≥1):  {np.mean(precisions):.3f}")
    print(f"\nNote: {WINNER} was already selected by automated metrics. "
          "This audit is diagnostic.")

Human relevance audit — 20 queries, 100 judgments
  nDCG@5:           0.592
  Graded P@5 (≥1):  0.330

Note: tfidf was already selected by automated metrics. This audit is diagnostic.


In [24]:
# ============ CELL 15: Lock chosen retriever + attach audit diagnostic ============
CHOSEN_RETRIEVER = RUNS_RETRIEVAL / "chosen_retriever.json"

# --- Human audit numbers (from Cell 14b) ---
HUMAN_AUDIT = {
    "n_queries": 20,
    "n_judgments": 100,
    "ndcg_at_5": 0.592,
    "graded_precision_at_5_ge1": 0.330,
    "note": (
        "Diagnostic only; does not affect winner selection. "
        "Intent-match Recall@5 overstates usefulness: only 1 in 3 "
        "retrieved docs is at least partially useful."
    ),
}

# --- Write the lock once ---
if CHOSEN_RETRIEVER.exists():
    existing = json.loads(CHOSEN_RETRIEVER.read_text(encoding="utf-8"))
    if existing["approach"] != WINNER:
        raise RuntimeError(
            f"chosen_retriever.json already exists with approach "
            f"'{existing['approach']}' but winner is '{WINNER}'. "
            f"Delete explicitly to re-choose."
        )
    print(f"chosen_retriever.json already locked: {existing['approach']}")

    # Attach / refresh the diagnostic block in place
    existing["human_audit_diagnostic"] = HUMAN_AUDIT
    CHOSEN_RETRIEVER.write_text(
        json.dumps(existing, indent=2), encoding="utf-8"
    )
    print("Refreshed human_audit_diagnostic block.")
else:
    chosen_payload = {
        "approach": WINNER,
        "metrics": CHOSEN_METRICS,
        "selection_rule": {
            "primary_metric": "recall@5",
            "tie_delta": 0.02,
            "tie_break": "simpler_or_cheaper",
            "precedence": precedence,
        },
        "candidate_count": len(candidates),
        "taxonomy_sha256":   tax_sha,
        "golden_sha256":     golden_sha,
        "classifier_sha256": chosen_sha,
        "corpus_sha256":     corpus_sha,
        "normalization_version": corpus.normalization_version,
        "locked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "note": "Immutable lock written before test evaluation.",
        "human_audit_diagnostic": HUMAN_AUDIT,
    }
    CHOSEN_RETRIEVER.write_text(
        json.dumps(chosen_payload, indent=2), encoding="utf-8"
    )
    print(f"chosen_retriever.json locked: {WINNER}")

# --- Recompute the SHA (audit block changes bytes) ---
chosen_retriever_sha = hashlib.sha256(CHOSEN_RETRIEVER.read_bytes()).hexdigest()
print(f"chosen_retriever SHA: {chosen_retriever_sha[:16]}...")

# --- Verify ---
_check = json.loads(CHOSEN_RETRIEVER.read_text(encoding="utf-8"))
assert _check["approach"] == WINNER
assert _check["human_audit_diagnostic"]["ndcg_at_5"] == 0.592
print("✓ Lock verified.")

chosen_retriever.json already locked: tfidf
Refreshed human_audit_diagnostic block.
chosen_retriever SHA: 33dbaea4a95ad148...
✓ Lock verified.


In [25]:
# ============ CELL 16: Load TEST (first read) ============
test_rows = []
with open(GOLDEN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["split"] == "test":
            test_rows.append(r)
test_df = pd.DataFrame(test_rows)

assert len(test_df) == golden_meta["test_size"]
assert set(test_df["root_id"].astype(int)) == TEST_ROOT_IDS
assert set(test_df["true_intent"].unique()) <= set(INTENT_NAMES)

# Leakage re-verification (defensive)
test_hashes = set(test_df["customer_text"].map(text_hash))
assert not (test_hashes & set(corpus.normalized))

print(f"Test rows: {len(test_df)}")
print(test_df["true_intent"].value_counts().to_string())

Test rows: 139
true_intent
on_board_issue        23
delay_compensation    18
service_disruption    18
other                 18
booking_issue         14
praise_or_chatter     11
refund_request        10
timetable_info        10
seat_reservation       9
lost_property          7
ambiguous              1


In [26]:
# ============ CELL 17: One-shot test evaluation ============
TEST_LOCK     = RUNS_RETRIEVAL / "TEST_LOCK.json"
FINAL_PREDS   = RUNS_RETRIEVAL / "predictions_test.jsonl"
FINAL_METRICS = RUNS_RETRIEVAL / "final_metrics.json"

if TEST_LOCK.exists():
    lock = json.loads(TEST_LOCK.read_text(encoding="utf-8"))
    if lock["chosen_sha"] != chosen_retriever_sha:
        raise RuntimeError(
            "Different chosen retriever — test was evaluated with another one. "
            "This violates the one-shot protocol."
        )
    # Verify existing artifacts match the lock
    assert hashlib.sha256(FINAL_PREDS.read_bytes()).hexdigest() == lock["predictions_sha256"]
    assert hashlib.sha256(FINAL_METRICS.read_bytes()).hexdigest() == lock["metrics_sha256"]
    assert hashlib.sha256((RUNS_RETRIEVAL / "corpus.jsonl").read_bytes()).hexdigest() == lock["corpus_sha256"]
    print("Test already frozen. No retrieval will be re-run.")
    raise SystemExit

print(f"First test evaluation. Chosen: {WINNER}")

# Rerun the winner on test
winner_retriever = {
    "tfidf": tfidf, "bm25": bm25, "embedding": emb,
    "hybrid_rrf": hybrid,
    "predicted_intent_bm25": pred_intent,
}[WINNER]

if WINNER == "predicted_intent_bm25":
    # Test-time predicted intents come from the frozen classifier predictions
    pred_test = pd.read_json(CLASSIFIER_PREDS, orient="records", lines=True)
    pred_map_test = dict(zip(pred_test["root_id"].astype(int), pred_test["pred_intent"]))
    def test_intent_resolver(q, *, root_id=None, true_intent=None):
        return pred_map_test.get(int(root_id), "other")
    runner = IntentConditionedRetriever(corpus, bm25, test_intent_resolver, name=WINNER)
    pq_test = evaluate_intent_conditioned(runner, test_df, corpus)
else:
    pq_test = evaluate_retriever_on_queries(winner_retriever, test_df, corpus)

agg_test = aggregate_metrics(pq_test)
rt_mix_test = response_type_mix(pq_test, corpus, k=5)

# Persist
pq_test.to_json(FINAL_PREDS, orient="records", lines=True, force_ascii=False)

final_payload = {
    "approach": WINNER,
    "chosen_sha": chosen_retriever_sha,
    "taxonomy_sha256": tax_sha,
    "golden_sha256": golden_sha,
    "corpus_sha256": corpus_sha,
    "metrics": agg_test,
    "response_type_mix_top5": rt_mix_test,
    "evaluated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
FINAL_METRICS.write_text(json.dumps(final_payload, indent=2), encoding="utf-8")

# Write lock
lock_payload = {
    "chosen_sha": chosen_retriever_sha,
    "corpus_sha256": hashlib.sha256((RUNS_RETRIEVAL / "corpus.jsonl").read_bytes()).hexdigest(),
    "predictions_sha256": hashlib.sha256(FINAL_PREDS.read_bytes()).hexdigest(),
    "metrics_sha256": hashlib.sha256(FINAL_METRICS.read_bytes()).hexdigest(),
    "evaluated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
TEST_LOCK.write_text(json.dumps(lock_payload, indent=2), encoding="utf-8")

print(f"\n=== TEST RESULTS ({WINNER}) ===")
for k, v in agg_test.items():
    if isinstance(v, float):
        print(f"  {k:20s} {v:.4f}")
print(f"\nGrounding diagnostics (top-5 response-type mix):")
for rt, frac in sorted(rt_mix_test.items(), key=lambda x: -x[1]):
    print(f"  {rt:20s} {frac:.3f}")

First test evaluation. Chosen: tfidf

=== TEST RESULTS (tfidf) ===
  recall@1             0.2014
  recall@1_cov         0.2137
  recall@3             0.3885
  recall@3_cov         0.4122
  recall@5             0.5324
  recall@5_cov         0.5649
  recall@10            0.7482
  recall@10_cov        0.7939
  mrr                  0.3419
  coverage             0.9424

Grounding diagnostics (top-5 response-type mix):
  informational        0.619
  resolution           0.374
  investigation        0.007


In [27]:
# ============ CELL 18: Freeze + manifest + handoff ============
ARTIFACT = RUNS_RETRIEVAL / "artifact_manifest.json"

files = []
for path in sorted(RUNS_RETRIEVAL.rglob("*")):
    if path.is_file() and path.name != "artifact_manifest.json":
        files.append({
            "path":       str(path.relative_to(PROJECT_ROOT)),
            "size_bytes": path.stat().st_size,
            "sha256":     hashlib.sha256(path.read_bytes()).hexdigest(),
        })

ARTIFACT.write_text(json.dumps({
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "taxonomy_sha256":   tax_sha,
    "golden_sha256":     golden_sha,
    "classifier_sha256": chosen_sha,
    "chosen_approach":   WINNER,
    "corpus_sha256":     corpus_sha,
    "normalization_version": corpus.normalization_version,
    "artifacts":         files,
}, indent=2), encoding="utf-8")

print(f"Wrote artifact_manifest.json ({len(files)} files)")

handoff = {
    "from_notebook": "05_retrieval.ipynb",
    "to_notebook":   "06_agent.ipynb",
    "consumes": [
        "runs/retrieval/predictions_test.jsonl",
        "runs/retrieval/final_metrics.json",
        "runs/retrieval/chosen_retriever.json",
        "runs/retrieval/corpus.jsonl",
    ],
    "does_not_rerun_retrieval": True,
    "verification_required_by_06": {
        "taxonomy_sha256":   tax_sha,
        "golden_sha256":     golden_sha,
        "classifier_sha256": chosen_sha,
        "retriever_sha":     chosen_retriever_sha,
        "corpus_sha256":     corpus_sha,
    },
    "notes": (
        "Corpus carries response_text and response_type for grounding. "
        "Top-5 per query is the recommended downstream K. Retrieval scores "
        "are NOT calibrated; treat them as ordinal."
    ),
}
(RUNS_RETRIEVAL / "handoff_to_06.json").write_text(
    json.dumps(handoff, indent=2), encoding="utf-8"
)
print("Handoff written: handoff_to_06.json")

Wrote artifact_manifest.json (27 files)
Handoff written: handoff_to_06.json


In [28]:
# ============ CELL 19: Failure analysis ============
pq_test = pd.read_json(FINAL_PREDS, orient="records", lines=True)

# Misses = queries where recall@5 == 0
misses = pq_test[pq_test["recall@5"] == 0].copy()
print(f"Queries with no relevant doc in top-5: {len(misses)} / {len(pq_test)}")

# Split: intent missing from corpus vs. retriever failed
intent_missing = misses[misses["corpus_coverage"] == 0]
retriever_failed = misses[misses["corpus_coverage"] == 1]
print(f"  Intent not in corpus: {len(intent_missing)}")
print(f"  Retriever missed:     {len(retriever_failed)}")

# Breakdown by intent
print("\nMisses by intent:")
print(pq_test.groupby("true_intent")["recall@5"].agg(["mean", "count"]).round(3).to_string())

# Save
misses[["root_id", "true_intent", "_source", "corpus_coverage", "top_ids"]].to_csv(
    RUNS_RETRIEVAL / "failure_analysis_test.csv", index=False, quoting=1, encoding="utf-8",
)
print("\nWrote failure_analysis_test.csv")

Queries with no relevant doc in top-5: 65 / 139
  Intent not in corpus: 8
  Retriever missed:     57

Misses by intent:
                     mean  count
true_intent                     
ambiguous           0.000      1
booking_issue       0.714     14
delay_compensation  0.500     18
lost_property       0.000      7
on_board_issue      0.522     23
other               0.222     18
praise_or_chatter   0.545     11
refund_request      0.200     10
seat_reservation    0.556      9
service_disruption  1.000     18
timetable_info      0.800     10

Wrote failure_analysis_test.csv


In [29]:
# ============ CELL 20: Deployment-scale sensitivity analysis ============
# Optional. Uses the FULL corpus (with classifier-derived intent labels for
# the additional docs). Report as a sensitivity analysis — its label quality
# is weaker than the primary benchmark.
print("Skipped by default. Enable only if you have time.")
print("This requires a second classifier pass over the full corpus.")
print("The label noise makes this a sensitivity analysis, not a headline result.")

Skipped by default. Enable only if you have time.
This requires a second classifier pass over the full corpus.
The label noise makes this a sensitivity analysis, not a headline result.
